# Compute daily mean wind speed

To align with DCPP model data, we will compute daily wind speed from daily mean u and v components.

- ERA5
- BARRA-R2
- BARRA-C2 later - not a reanalysis as no DA

In [1]:
from dask.distributed import Client,LocalCluster
from dask_jobqueue import PBSCluster

In [2]:
PROJECT = "dt6"

In [3]:
walltime = "00:30:00"
cores = 48
memory = str(4 * cores) + "GB"

cluster = PBSCluster(
    walltime=str(walltime),
    cores=cores,
    memory=str(memory),
    processes=cores,
    job_extra_directives=[
        "-q normal",
        "-P "+PROJECT,
        "-l ncpus="+str(cores),
        "-l mem="+str(memory),
        "-l storage=gdata/xp65+gdata/w42+scratch/w42+gdata/gb02+scratch/gb02+gdata/ng72+scratch/ng72+gdata/rt52"
    ],
    local_directory="$TMPDIR",
    job_directives_skip=["select"],
    log_directory="/scratch/w42/dr6273/tmp/logs"
)

In [4]:
cluster.scale(jobs=1)
client = Client(cluster)

In [5]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.6.121.4:34861,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [6]:
import xarray as xr

In [8]:
%cd /g/data/w42/dr6273/work/wind_drought/
import functions as fn

%load_ext autoreload
%autoreload 2

/g/data/w42/dr6273/work/wind_drought


In [9]:
ERA_PATH = "/g/data/rt52/era5/single-levels/reanalysis/" # hourly instantaneous
# BARRA_R2_PATH = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/" # daily mean (?)

WRITE_PATH = "/g/data/ng72/dr6273/work/projects/wind_drought/data/"

# Compute wind speed by year

In [12]:
def era_preprocess(ds):
    """
    Preprocess function for open_mfdataset.
    Selects Australian region, computes daily mean and renames coords.
    """
    ds = ds.sel(
        longitude=slice(REGION[0], REGION[1]),
        latitude=slice(REGION[2], REGION[3])
    )
    ds = ds.rename(
        {'longitude': 'lon',
         'latitude': 'lat'}
    )
    ds = ds.coarsen(time=24).mean()
    ds["time"] = ds["time"].dt.floor("D")
    ds = ds.chunk({"time": -1, "lat": -1, "lon": -1})
    return ds

In [13]:
def load_hourly_era5(preprocess, variable, year, data_path=ERA_PATH):
    """
    Load and preprocess hourly data for a given year
    
    preprocess: preprocess function
    variable: name of variable to process
    year: year to process
    data_path: path to hourly data
    """
    # Open all hours in the year (~33 GB)
    hourly = xr.open_mfdataset(
        data_path + variable + "/" + str(year) + "/*.nc",
        preprocess=preprocess
    )
    return hourly

In [15]:
REGION = fn.get_Aus_boundary()

In [16]:
YEARS = range(1940, 2025)

In [17]:
def era_windspeed(u_name, v_name, ws_name, years, write_path):
    """
    Compute wind speed from daily mean u and v winds.

    u_name, v_name: str, names of u and v variables in ERA5
    ws_name: str, desired wind speed variable name
    years: range, years to process
    write_path: str, path to write directory
    """
    da_list = []
    for year in years:
        if year % 5 == 0:
            print(year)
            
        u = load_hourly_era5(
            era_preprocess,
            u_name,
            year
        )
    
        v = load_hourly_era5(
            era_preprocess,
            v_name,
            year
        )
        
        ws = fn.windspeed(
            u.rename({"u"+u_name[:-1]: ws_name}),
            v.rename({"v"+v_name[:-1]: ws_name})
        )
        # ws = ws.chunk({"time": -1})
        
        # encoding = {
        #     ws_name: {"dtype": "float32"}
        # }
        # w100.to_netcdf(
        #     write_path + "100w_era5_hourly_" + str(year) + "_Aus.nc",
        #     mode="w",
        #     encoding=encoding
        # )
        da_list.append(ws)
        
    ws = xr.concat(da_list, dim="time")
    ws = ws.chunk({"time": "300MB"})
        
    encoding = {
        ws_name: {"dtype": "float32"}
    }
    ws.to_zarr(
        write_path + ws_name + "_era5_daily_Aus.zarr",
        consolidated=True,
        mode="w",
        zarr_format=2, # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
        encoding=encoding
    )

In [18]:
era_windspeed("100u", "100v", "ws100m", YEARS, WRITE_PATH)

1940
1945
1950
1955
1960
1965
1970
1975
1980
1985
1990
1995
2000
2005
2010
2015
2020


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 54.90 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [19]:
SFC_YEARS = range(1979, 2025)

In [20]:
era_windspeed("10u", "10v", "ws10m", SFC_YEARS, WRITE_PATH)

1980
1985
1990
1995
2000
2005
2010
2015
2020


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 29.23 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


# Close cluster

In [21]:
client.close()
cluster.close()

INFO:dask_jobqueue.pbs:Resource specification for PBS not set, initializing it to select=1:ncpus=48:mem=179GB
